In [ ]:
import json
import itertools
import networkx as nx
import pandas as pd
import random
from collections import defaultdict

# --- FUNGSI PEMUATAN DAN PERSIAPAN DATA ---

def load_json_file(filename: str) -> dict | list:
    """Membuka dan memuat data dari sebuah file JSON."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: File '{filename}' tidak ditemukan.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Gagal mem-parsing JSON dari file '{filename}'.")
        return None

def build_network_graph(network_data: dict) -> nx.Graph:
    """Membangun graf jaringan dari data networkDefinition.json."""
    G = nx.Graph()
    if not network_data: return G
    for entity in network_data.get('entity', []):
        G.add_node(entity['id'], **entity)
    for link in network_data.get('link', []):
        G.add_edge(link['s'], link['d'], PR=link['PR'], BW=link['BW'])
    return G

def build_placement_matrix(placement_data: dict) -> dict:
    """Membangun kamus penempatan dari data allocDefinition.json."""
    placement_matrix = defaultdict(list)
    if not placement_data: return {}
    for allocation in placement_data.get('initialAllocation', []):
        module_name = allocation["module_name"]
        resource_id = allocation["id_resource"]
        if resource_id not in placement_matrix[module_name]:
            placement_matrix[module_name].append(resource_id)
    return dict(placement_matrix)

# --- FUNGSI PERHITUNGAN DELAY (Sama seperti sebelumnya) ---

def _calculate_path_delay(path: list, graph: nx.Graph, packet_size: float) -> float:
    """Menghitung total delay di sepanjang sebuah path."""
    total_delay = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        link_data = graph.get_edge_data(u, v)
        if link_data:
            bw_ms = link_data.get('BW', 1) / 1000.0
            pr_ms = link_data.get('PR', 0)
            transmission_delay = packet_size / bw_ms if bw_ms > 0 else float('inf')
            link_delay = pr_ms + transmission_delay
            total_delay += link_delay
    return total_delay

def calculate_adra(start_node: int, end_node: int, graph: nx.Graph, packet_size: float) -> float:
    """Menghitung ADRA: Delay dari requester ke titik masuk aplikasi."""
    try:
        path = nx.shortest_path(graph, source=start_node, target=end_node)
        return _calculate_path_delay(path, graph, packet_size)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return float('inf')

def calculate_adsa(target_app: dict, placement_matrix: dict, network_graph: nx.Graph, packet_size: float) -> float:
    """Menghitung ADSA: Rata-rata delay antar layanan dalam satu aplikasi."""
    service_list = [module['name'] for module in target_app.get('module', [])]
    all_device_locations = set()
    for service_name in service_list:
        if service_name in placement_matrix:
            all_device_locations.update(placement_matrix[service_name])
    if len(all_device_locations) < 2: return 0.0
    services_delay_list = []
    for dev1, dev2 in itertools.combinations(list(all_device_locations), 2):
        delay = calculate_adra(dev1, dev2, network_graph, packet_size)
        if delay != float('inf'):
            services_delay_list.append(delay)
    if not services_delay_list: return 0.0
    return sum(services_delay_list) / len(services_delay_list)

# --- BLOK EKSEKUSI UTAMA YANG DIMODIFIKASI ---
if __name__ == "__main__":
    # 1. Muat semua data dari file JSON
    network_def = load_json_file('networkDefinition.json')
    app_def = load_json_file('appDefinition.json')
    alloc_def = load_json_file('allocDefinition.json')

    if not all([network_def, app_def, alloc_def]):
        print("Satu atau lebih file definisi tidak dapat dimuat. Program berhenti.")
    else:
        # 2. Persiapkan struktur data
        network_graph = build_network_graph(network_def)
        placement_matrix = build_placement_matrix(alloc_def)
        
        # Siapkan daftar node untuk requester acak (kecuali node cloud ID 100)
        potential_requesters = [node['id'] for node in network_def.get('entity', []) if node.get('type') != 'CLOUD']
        if not potential_requesters:
            potential_requesters = [node for node in network_graph.nodes() if node != 100]

        # 3. Loop untuk semua aplikasi dan simpan hasilnya
        results_list = []
        PACKET_SIZE_BITS = 1_500_000  # Ukuran paket dalam bit (1.5 MB)

        for app_data in app_def:
            app_id = app_data['name']
            
            # Hitung ADSA
            adsa_result = calculate_adsa(app_data, placement_matrix, network_graph, PACKET_SIZE_BITS)
            
            # Hitung ADRA dengan requester acak
            entry_message = f"M.USER.APP.{app_id}"
            entry_module_name = next((t.get("module") for t in app_data.get("transmission", []) if t.get("message_in") == entry_message), None)
            
            adra_result = 0.0
            if entry_module_name and entry_module_name in placement_matrix and potential_requesters:
                entry_node = placement_matrix[entry_module_name][0]
                random_requester_id = random.choice(potential_requesters)
                adra_result = calculate_adra(random_requester_id, entry_node, network_graph, PACKET_SIZE_BITS)
            
            # Simpan hasil
            results_list.append({
                'app_id': app_id,
                'adra_ms': adra_result if adra_result != float('inf') else None,
                'adsa_ms': adsa_result,
                'total_delay_ms': (adra_result + adsa_result) if adra_result != float('inf') else None
            })

        # 4. Buat DataFrame dan lakukan analisis
        df_results = pd.DataFrame(results_list)
        df_results.set_index('app_id', inplace=True)

        # convert delay to seconds
        df_results['adra_s'] = df_results['adra_ms'] / 1000 if df_results['adra_ms'] is not None else None
        df_results['adsa_s'] = df_results['adsa_ms'] / 1000 if df_results['adsa_ms'] is not None else None
        df_results['total_delay_s'] = df_results['total_delay_ms'] / 1000 if df_results['total_delay_ms'] is not None else None
        
        print("--- Rekapitulasi Hasil Delay untuk Semua Aplikasi ---")
        print(df_results.to_string(float_format="%.2f"))
        print("\n" + "="*50 + "\n")

        print("--- Analisis Rata-rata Delay ---")
        
        # calculate average from adra_ms and adsa_ms
        average_adra = df_results['adra_ms'].mean()
        average_adra_s = average_adra / 1000 if average_adra is not None else None
        average_adsa = df_results['adsa_ms'].mean()
        average_adsa_s = average_adsa / 1000 if average_adsa is not None else None
        print(f"Rata-rata ADRA: {average_adra:.2f} ms ({average_adra_s:.2f} s)")
        print(f"Rata-rata ADSA: {average_adsa:.2f} ms ({average_adsa_s:.2f} s)")

        # Menghitung rata-rata keseluruhan dengan mengabaikan nilai None (inf)
        average_total_delay = df_results['total_delay_ms'].mean()
        print(f"Rata-rata Total Delay (ADRA + ADSA) untuk semua aplikasi: {average_total_delay:.2f} ms")
        
        # Aplikasi dengan delay terendah dan tertinggi
        app_min_delay = df_results['total_delay_ms'].idxmin()
        min_delay = df_results['total_delay_ms'].min()
        app_max_delay = df_results['total_delay_ms'].idxmax()
        max_delay = df_results['total_delay_ms'].max()
        
        print(f"Aplikasi dengan delay terendah: App '{app_min_delay}' ({min_delay:.2f} ms)")
        print(f"Aplikasi dengan delay tertinggi: App '{app_max_delay}' ({max_delay:.2f} ms)")

        

--- Rekapitulasi Hasil Delay untuk Semua Aplikasi ---
         adra_ms  adsa_ms  total_delay_ms  adra_s  adsa_s  total_delay_s
app_id                                                                  
0       80020.00 52011.00       132031.00   80.02   52.01         132.03
1       60015.00 60742.09       120757.09   60.02   60.74         120.76
2       40010.00 41342.33        81352.33   40.01   41.34          81.35
3       40010.00 34674.00        74684.00   40.01   34.67          74.68
4       60015.00 49154.57       109169.57   60.02   49.15         109.17
5       40010.00 34674.00        74684.00   40.01   34.67          74.68
6       60015.00 50678.67       110693.67   60.02   50.68         110.69
7       60015.00 42010.17       102025.17   60.02   42.01         102.03
8       60015.00 34674.00        94689.00   60.02   34.67          94.69
9       60015.00 52011.00       112026.00   60.02   52.01         112.03
10      20005.00 49345.22        69350.22   20.00   49.35          69.

In [10]:
if __name__ == "__main__":
    # 1. Muat semua data dari file JSON
    network_def = load_json_file('networkDefinition.json')
    app_def = load_json_file('appDefinition.json')
    alloc_def = load_json_file('allocDefinitionILP.json')

    if not all([network_def, app_def, alloc_def]):
        print("Satu atau lebih file definisi tidak dapat dimuat. Program berhenti.")
    else:
        # 2. Persiapkan struktur data
        network_graph = build_network_graph(network_def)
        placement_matrix = build_placement_matrix(alloc_def)
        
        # Siapkan daftar node untuk requester acak (kecuali node cloud ID 100)
        potential_requesters = [node['id'] for node in network_def.get('entity', []) if node.get('type') != 'CLOUD']
        if not potential_requesters:
            potential_requesters = [node for node in network_graph.nodes() if node != 100]

        # 3. Loop untuk semua aplikasi dan simpan hasilnya
        results_list = []
        PACKET_SIZE_BITS = 1_500_000  # Ukuran paket dalam bit (1.5 MB)

        for app_data in app_def:
            app_id = app_data['name']
            
            # Hitung ADSA
            adsa_result = calculate_adsa(app_data, placement_matrix, network_graph, PACKET_SIZE_BITS)
            
            # Hitung ADRA dengan requester acak
            entry_message = f"M.USER.APP.{app_id}"
            entry_module_name = next((t.get("module") for t in app_data.get("transmission", []) if t.get("message_in") == entry_message), None)
            
            adra_result = 0.0
            if entry_module_name and entry_module_name in placement_matrix and potential_requesters:
                entry_node = placement_matrix[entry_module_name][0]
                random_requester_id = random.choice(potential_requesters)
                adra_result = calculate_adra(random_requester_id, entry_node, network_graph, PACKET_SIZE_BITS)
            
            # Simpan hasil
            results_list.append({
                'app_id': app_id,
                'adra_ms': adra_result if adra_result != float('inf') else None,
                'adsa_ms': adsa_result,
                'total_delay_ms': (adra_result + adsa_result) if adra_result != float('inf') else None
            })

        # 4. Buat DataFrame dan lakukan analisis
        df_results = pd.DataFrame(results_list)
        df_results.set_index('app_id', inplace=True)

        # convert delay to seconds
        df_results['adra_s'] = df_results['adra_ms'] / 1000 if df_results['adra_ms'] is not None else None
        df_results['adsa_s'] = df_results['adsa_ms'] / 1000 if df_results['adsa_ms'] is not None else None
        df_results['total_delay_s'] = df_results['total_delay_ms'] / 1000 if df_results['total_delay_ms'] is not None else None
        
        print("--- Rekapitulasi Hasil Delay untuk Semua Aplikasi ---")
        print(df_results.to_string(float_format="%.2f"))
        print("\n" + "="*50 + "\n")

        print("--- Analisis Rata-rata Delay ---")
        
        # calculate average from adra_ms and adsa_ms
        average_adra = df_results['adra_ms'].mean()
        average_adra_s = average_adra / 1000 if average_adra is not None else None
        average_adsa = df_results['adsa_ms'].mean()
        average_adsa_s = average_adsa / 1000 if average_adsa is not None else None
        print(f"Rata-rata ADRA: {average_adra:.2f} ms ({average_adra_s:.2f} s)")
        print(f"Rata-rata ADSA: {average_adsa:.2f} ms ({average_adsa_s:.2f} s)")

        # Menghitung rata-rata keseluruhan dengan mengabaikan nilai None (inf)
        average_total_delay = df_results['total_delay_ms'].mean()
        print(f"Rata-rata Total Delay (ADRA + ADSA) untuk semua aplikasi: {average_total_delay:.2f} ms")
        
        # Aplikasi dengan delay terendah dan tertinggi
        app_min_delay = df_results['total_delay_ms'].idxmin()
        min_delay = df_results['total_delay_ms'].min()
        app_max_delay = df_results['total_delay_ms'].idxmax()
        max_delay = df_results['total_delay_ms'].max()
        
        print(f"Aplikasi dengan delay terendah: App '{app_min_delay}' ({min_delay:.2f} ms)")
        print(f"Aplikasi dengan delay tertinggi: App '{app_max_delay}' ({max_delay:.2f} ms)")


--- Rekapitulasi Hasil Delay untuk Semua Aplikasi ---
        adra_ms  adsa_ms  total_delay_ms  adra_s  adsa_s  total_delay_s
app_id                                                                 
0          0.00 48010.67        48010.67    0.00   48.01          48.01
1      20005.00 61348.44        81353.44   20.00   61.35          81.35
2      80020.00 48010.67       128030.67   80.02   48.01         128.03
3      40010.00 36008.00        76018.00   40.01   36.01          76.02
4      60015.00 49154.57       109169.57   60.02   49.15         109.17
5      60015.00 41342.33       101357.33   60.02   41.34         101.36
6      60015.00 49440.43       109455.43   60.02   49.44         109.46
7      60015.00 47746.99       107761.99   60.02   47.75         107.76
8      80020.00 42295.79       122315.79   80.02   42.30         122.32
9      60015.00 46810.90       106825.90   60.02   46.81         106.83
10     80020.00 52921.98       132941.98   80.02   52.92         132.94
11     600